# 第十讲演示：Python数据分析（NumPy与Pandas进阶）

配套课件：`ch10_numpy_pandas_advanced.md`（36页） ｜ 教案：`lesson_plans/教案_Python数据分析_NumPy与Pandas进阶_2026-11-12.md` ｜ 演示数据：`demo_data/expr_matrix.csv` + `demo_data/lung_clinical.csv`

结构：单元格与课件页码一一对应；标注 **【课后自学】** 的单元格对应课件"课后自学"页（数组创建、显式广播、transform），课上不投屏。

In [1]:
# ── 环境自检 (Environment Check) ──────────────────────────────────────────
import math
import numpy as np
import pandas as pd

print("python OK | numpy", np.__version__, "| pandas", pd.__version__)

python OK | numpy 2.5.4 | pandas 3.1.1


## Slide 5 · 第9讲作业回顾：分步写法 vs 方法链

In [2]:
# ── 读入真实数据（后续所有演示共用） ─────────────────────────────────────
# 路径锚定到本Notebook所在目录：无论从哪里启动Jupyter都能找到数据
from pathlib import Path
data_path = Path.cwd()
while not (data_path / "demo_data" / "expr_matrix.csv").exists():
    if data_path.parent == data_path:
        raise FileNotFoundError("找不到 demo_data/expr_matrix.csv")
    data_path = data_path.parent

expr_df = pd.read_csv(data_path / "demo_data" / "expr_matrix.csv")
lung = pd.read_csv(data_path / "demo_data" / "lung_clinical.csv")
print("expr_matrix:", expr_df.shape, "| lung:", lung.shape)

expr_matrix: (28, 21) | lung: (30, 9)


In [3]:
# ── 分步 vs 方法链：R 的 %>% ↔ Python 的 . ─────────────────────────────
# Slide 5 · 同一件事，链着写：先筛选、再选列、最后加列
sub = lung[lung["age"] >= 60]                                # filter
sub = sub[["patient_id", "age", "egfr_expr"]]                # select
sub = sub.assign(log2_expr=np.log2(sub["egfr_expr"] + 1))    # mutate
print("分步三步:", sub.shape)

chained = (lung[lung["age"] >= 60]
           [["patient_id", "age", "egfr_expr"]]
           .assign(log2_expr=lambda d: np.log2(d["egfr_expr"] + 1)))
print("方法链一条:", chained.shape, "| 与分步一致:", chained.equals(sub))


分步三步: (18, 4)
方法链一条: (18, 4) | 与分步一致: True


## Slide 8 · ndarray：相当于 R 的原子向量(atomic vector)

In [4]:
# ── ndarray vs list：整组一起算 vs 逐个处理 ─────────────────────────────
# Slide 8 · 强调与R的对照（右侧注释）；np.log2 一行算全组
counts = [8, 3, 17]           # list —— R: c(8, 3, 17)
arr = np.array([8, 3, 17])    # ndarray —— 同质类型

print("arr * 2          :", arr * 2)      # array([16,  6, 34]) 向量化
print("counts * 2       :", counts * 2)   # 列表乘法 = 重复拼接 ⚠️ 不是翻倍！

tpm = np.array([120.0, 30.0, 480.0, 8.0])
print("np.log2(tpm + 1) :", np.round(np.log2(tpm + 1), 2))   # R: log2(tpm + 1)

arr * 2          : [16  6 34]
counts * 2       : [8, 3, 17, 8, 3, 17]
np.log2(tpm + 1) : [6.92 4.95 8.91 3.17]


## Slide 9 · 性能实测：向量化到底快多少？

In [5]:
# ── 计时对比：列表推导式 vs np.log2 ─────────────────────────────────────
# Slide 9 · 实测参考（2026-11-10 本机）：~61 ms vs ~5.3 ms ≈ 12×；课堂以现场输出为准
# 课件上讲 %%timeit 单元格魔法的三格写法；本格用 timeit 模块等价实现，便于一键重跑
import timeit
big = np.random.rand(1_000_000)
t_loop = timeit.timeit(lambda: [math.log2(x + 1) for x in big], number=5) / 5
t_vec  = timeit.timeit(lambda: np.log2(big + 1), number=50) / 50
print(f"逐元素循环: {t_loop*1000:.1f} ms | 向量化: {t_vec*1000:.2f} ms")
print(f"向量化快约 {t_loop / t_vec:.0f} 倍  （书：一般10–100倍，越接近纯乘法越逼近上限）")

逐元素循环: 62.5 ms | 向量化: 5.34 ms
向量化快约 12 倍  （书：一般10–100倍，越接近纯乘法越逼近上限）


## Slide 10 · Z-score标准化：在真实表达矩阵上算（第12–13讲的输入）

In [6]:
# ── Z-score：直接在表达矩阵 DataFrame 上算（第12–13讲热图输入） ──────────
# Slide 10 · 与第13讲逐字一致：expr - expr.mean() 按列对齐自动铺开（广播的Pandas形态）
expr = pd.read_csv(data_path / "demo_data" / "expr_matrix.csv",
                   index_col="patient_id")   # 28×20：行=患者 列=基因（与第13讲同名同读法）
print("expr.shape:", expr.shape)

z = (expr - expr.mean()) / expr.std()   # 按基因（列）标准化
print("z.iloc[:4, :4]:")
print(z.iloc[:4, :4].round(2).to_string())
print("各列均值最大偏差:", f"{np.abs(z.mean()).max():.2e}", "—— 每个基因均值≈0")
print("z 范围:", round(z.min().min(), 2), "~", round(z.max().max(), 2), "（第13讲课件宣称值 -1.13 ~ 4.73）")
print("口径: pandas .std() 默认 ddof=1（与R的sd同口径）；第13讲 seaborn z_score 用 ddof=0，视觉无差")

expr.shape: (28, 20)
z.iloc[:4, :4]:
            EGFR  KRAS   ALK  TP53
patient_id                        
LC001       1.14 -0.62 -0.38 -0.93
LC002      -0.56 -0.23  2.43 -0.68
LC004       0.34  0.35  0.38 -0.45
LC005      -0.46 -0.17 -0.43  1.83
各列均值最大偏差: 2.34e-16 —— 每个基因均值≈0
z 范围: -1.13 ~ 4.73 （第13讲课件宣称值 -1.13 ~ 4.73）
口径: pandas .std() 默认 ddof=1（与R的sd同口径）；第13讲 seaborn z_score 用 ddof=0，视觉无差


## 【课后自学】数组创建、核心属性、显式广播

In [7]:
# ── 【课后自学】数组创建、核心属性、显式广播 ────────────────────────────
# 课后自学 · 对应 McKinney ch4 §4.1–4.2 与附录A；课上只指地图，不展开
print("arange(0, 10, 2) :", np.arange(0, 10, 2))     # R: seq(0, 8, by=2)，含头不含尾
print("linspace(0, 1, 5) :", np.linspace(0, 1, 5))  # 等分取5个点
print("zeros((2,3))      :\n", np.zeros((2, 3)))
print("ones(4)           :", np.ones(4))

rng = np.random.default_rng(42)
expr_demo = rng.random((4, 3))    # 4基因×3样本
print("random (4,3)      :\n", np.round(expr_demo, 2))
print("shape:", expr_demo.shape, "| dtype:", expr_demo.dtype,
      "| ndim:", expr_demo.ndim, "| size:", expr_demo.size)

expr_np = np.array([[8.5, 2.1, 9.7],
                    [3.2, 5.6, 4.1],
                    [7.4, 6.9, 7.9],
                    [5.5, 5.8, 5.2]])   # 4基因×3样本
row_means = expr_np.mean(axis=1)
centered = expr_np - row_means[:, None]  # 显式广播：[:, None] 把行均值"立起来"铺到每列
print("\nNumPy 显式广播（DataFrame 会自动对齐，本讲用不上）:\n", np.round(centered, 2))

arange(0, 10, 2) : [0 2 4 6 8]
linspace(0, 1, 5) : [0.   0.25 0.5  0.75 1.  ]
zeros((2,3))      :
 [[0. 0. 0.]
 [0. 0. 0.]]
ones(4)           : [1. 1. 1. 1.]
random (4,3)      :
 [[0.77 0.44 0.86]
 [0.7  0.09 0.98]
 [0.76 0.79 0.13]
 [0.45 0.37 0.93]]
shape: (4, 3) | dtype: float64 | ndim: 2 | size: 12

NumPy 显式广播（DataFrame 会自动对齐，本讲用不上）:
 [[ 1.73 -4.67  2.93]
 [-1.1   1.3  -0.2 ]
 [-0.   -0.5   0.5 ]
 [ 0.    0.3  -0.3 ]]


## Slide 12 · 向量表达式写得出就整列算；写不出就用 .map

In [8]:
# ── ① 向量表达式（整列算） vs ② 写不出时用 .map ─────────────────────────
# Slide 12 · 左：np.log2 整列；右：没有 np.late_stage，分期规则用 map
lg = lung[["patient_id", "egfr_expr", "stage"]].copy()

# ① 写得出：向量表达式（没有 for / if）
lg["log2_egfr"] = np.log2(lg["egfr_expr"] + 1)

# ② 写不出：没有 np.late_stage(lg["stage"]) —— 只能写 Python 逻辑再 map
def stage_label(stage):
    return "晚期" if stage.startswith("III") or stage == "IV" else "早期"

lg["stage_group"] = lg["stage"].map(stage_label)  # R: sapply()
print(lg.head(4).to_string())
assert lg["stage_group"].notna().all(), "stage_group 不应有缺失"
print("\nSeries.map(f) 与 .apply(f) 元素级等价；第13讲 row_colors=stage.map(颜色字典) 用的就是它")

  patient_id  egfr_expr stage  log2_egfr stage_group
0      LC001       8.52  IIIB   3.250962          晚期
1      LC002       2.14    II   1.650765          早期
2      LC003       9.71    IV   3.420887          晚期
3      LC004       7.88  IIIA   3.150560          晚期

Series.map(f) 与 .apply(f) 元素级等价；第13讲 row_colors=stage.map(颜色字典) 用的就是它


## Slide 13 · 链式索引与Copy-on-Write：能赋值 ≠ 赋上了

In [9]:
# ── CoW（Copy-on-Write，pandas 3.x 默认）：切片即副本 ────────────────────
# Slide 13 · 先切片存变量再赋列=安全；真·链式赋值=只警告、不生效
sub2 = lung[lung["age"] >= 60]
sub2["age_group"] = "senior"                      # ①② 先切片存变量，再赋新列
print("① 切片存变量再赋列:", sub2.shape[0], "行 | lung 本体受影响:", "age_group" in lung.columns)

import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    lung[lung["age"] >= 60]["age_group2"] = "senior"   # ③ 真·链式赋值 ⚠️
print("③ 真·链式赋值: 警告", len(w), "条 | 赋值生效:", "age_group2" in lung.columns)
print("   口诀：切片后要改就 .copy()；一步到位 df.loc[cond, col] = ...")


① 切片存变量再赋列: 18 行 | lung 本体受影响: False
③ 真·链式赋值: 警告 1 条 | 赋值生效: False
   口诀：切片后要改就 .copy()；一步到位 df.loc[cond, col] = ...


## Slide 17 · merge：表达矩阵 × 临床信息

In [10]:
# ── merge 四种连接方式：先数行数，再下结论 ────────────────────────────────
# Slide 17 · expr_matrix 缺 LC003/LC027（测序QC剔除），行数变化是真实教学点
for how in ("inner", "left", "right", "outer"):
    merged = pd.merge(expr_df, lung, on="patient_id", how=how)
    print(f"expr←lung  how={how:5s}: {len(merged):2d} 行")
print()
for how in ("inner", "left"):
    merged = pd.merge(lung, expr_df, on="patient_id", how=how)
    print(f"lung←expr  how={how:5s}: {len(merged):2d} 行")

expr←lung  how=inner: 28 行
expr←lung  how=left : 28 行
expr←lung  how=right: 30 行
expr←lung  how=outer: 30 行

lung←expr  how=inner: 28 行
lung←expr  how=left : 30 行


In [11]:
# ── 左表决定了谁被保留：临床为主表就用 lung ← expr ──────────────────────
# Slide 17 · merge（续）：合并后纳入30例，2例因测序QC无表达数据——以透明、可追溯的方式报告
merged = pd.merge(lung, expr_df, on="patient_id", how="left")
n_no_expr = merged["EGFR"].isnull().sum()
print("左表=lung（临床为主）:", len(merged), "行；无表达数据的样本:", n_no_expr, "个")
print(merged.loc[merged["EGFR"].isnull(), ["patient_id", "stage", "EGFR"]].to_string(index=False))

左表=lung（临床为主）: 30 行；无表达数据的样本: 2 个
patient_id stage  EGFR
     LC003    IV   NaN
     LC027  IIIB   NaN


> 💬 **讨论点**：论文写"合并后纳入30例，其中2例因测序质控无表达数据"——这就是'先数行数'习惯的操作化。

## Slide 19 · concat：按位置拼接

In [12]:
# ── concat：按位置拼接（axis=0 行 / axis=1 列） ─────────────────────────
# Slide 19 · 一句话区分：merge 按键找朋友，concat 按位置拼积木
batch_a, batch_b = lung.iloc[:15], lung.iloc[15:]        # 模拟两个批次
print("axis=0 行拼接:", pd.concat([batch_a, batch_b], axis=0).shape, "—— R: rbind()")
print("axis=1 列拼接:", pd.concat([expr.iloc[:, :10], expr.iloc[:, 10:]], axis=1).shape,
      "—— R: cbind()（按索引对齐）")

axis=0 行拼接: (30, 9) —— R: rbind()
axis=1 列拼接: (28, 20) —— R: cbind()（按索引对齐）


## Slide 20 · reindex：按索引标签对齐

In [13]:
# ── reindex：按索引标签对齐取数 ──────────────────────────────────────────
# Slide 20 · 第13讲聚类热图：把 stage 对齐到表达矩阵 28 个样本
stage = lung.set_index("patient_id")["stage"]       # 先把 ID 设为索引
aligned = stage.reindex(expr_df["patient_id"])      # 按标签对齐：28行全对上
print("set_index 后 reindex:", len(aligned), "行 | 缺失:", aligned.isna().sum())

trap = lung["stage"].reindex(expr_df["patient_id"]) # ⚠️ 忘了 set_index
print("忘 set_index 的陷阱  : 缺失", trap.isna().sum(), "/", len(trap),
      "—— 整列NaN，安静不报错")


set_index 后 reindex: 28 行 | 缺失: 0
忘 set_index 的陷阱  : 缺失 28 / 28 —— 整列NaN，安静不报错


## Slide 22 · groupby + agg：拆分-应用-合并

In [14]:
# ── 分组聚合：对照 R 的 group_by + summarise ────────────────────────────
# Slide 22 · named agg：列名直接起好
summary = (lung.groupby("stage", as_index=False)
               .agg(n=("patient_id", "count"),
                    med_surv=("survival_months", "median")))
print(summary.to_string(index=False))

stage  n  med_surv
    I  3     58.60
   II  8     42.90
 IIIA  4     28.05
 IIIB  6     15.95
   IV  9      5.90


## Slide 23 · 真数据跑一遍：分期×吸烟的中位生存

In [15]:
# ── 多列分组 + 多聚合 ────────────────────────────────────────────────────
# Slide 23 · 表格与课件一致：7组；IIIA/yes 组只有1人，样本量小不下结论
summary2 = (lung.groupby(["stage", "smoking"], as_index=False)
                .agg(n=("patient_id", "count"),
                     med_surv=("survival_months", "median"),
                     med_expr=("egfr_expr", "median")))
print(summary2.to_string(index=False))

stage smoking  n  med_surv  med_expr
    I      no  3     58.60     3.020
   II      no  5     42.90     3.950
   II     yes  3     39.95     5.710
 IIIA      no  3     29.80     4.660
 IIIA     yes  1     24.90     7.880
 IIIB     yes  6     15.95     7.555
   IV     yes  9      5.90     9.930


## Slide 24 · MultiIndex太碍事？as_index=False

In [16]:
# ── 有无 as_index 的对比 ─────────────────────────────────────────────────
# Slide 24 · 初学统一 as_index=False：打印好看、取值简单、存CSV不乱
multi = lung.groupby(["stage", "smoking"]).agg(
    n=("patient_id", "count"))
print("默认 MultiIndex 索引:", multi.index.names, "| 取值要元组:", multi.loc[("II", "yes"), "n"])
flat = lung.groupby(["stage", "smoking"], as_index=False).agg(
    n=("patient_id", "count"))
print("as_index=False 列:", list(flat.columns))

默认 MultiIndex 索引: ['stage', 'smoking'] | 取值要元组: 3
as_index=False 列: ['stage', 'smoking', 'n']


## 【课后自学】transform：组内统计，行数不变

In [17]:
# ── agg 变矮 vs transform 不变 ───────────────────────────────────────────
# 【课后自学】transform · 用 shape 当场验证行数
agg_h = lung.groupby("stage").agg(med=("survival_months", "median"))
print("agg 后:", agg_h.shape, "—— 每组一行")
out2 = lung.copy()
out2["stage_med"] = lung.groupby("stage")["survival_months"].transform("median")
out2["surv_ratio"] = out2["survival_months"] / out2["stage_med"]
print("transform 后:", out2.shape, "—— 行数不变")
print(out2[["patient_id", "stage", "survival_months", "stage_med", "surv_ratio"]].head(5).to_string(index=False))

agg 后: (5, 1) —— 每组一行
transform 后: (30, 11) —— 行数不变
patient_id stage  survival_months  stage_med  surv_ratio
     LC001  IIIB             18.3      15.95    1.147335
     LC002    II             41.2      42.90    0.960373
     LC003    IV              7.6       5.90    1.288136
     LC004  IIIA             24.9      28.05    0.887701
     LC005     I             55.8      58.60    0.952218


## Slide 26 · 长宽转换：melt 融化 = 宽变长

In [18]:
# ── 长宽转换：melt融化=宽变长 ────────────────────────────────────────────
# Slide 26 · R第5讲 reshape2::melt 同名同义；tidyverse: pivot_longer
mini = expr_df[["patient_id", "EGFR", "KRAS", "TP53"]].head(2)   # 2样本小表，投影可见
long = mini.melt(id_vars="patient_id",
                 value_vars=["EGFR", "KRAS", "TP53"],
                 var_name="gene", value_name="tpm")
print("melt 后:", long.shape, "—— 2样本×3基因 = 6行")
print(long.head(6).to_string(index=False))

wide = long.pivot(index="patient_id", columns="gene", values="tpm")
print("\npivot 回宽:", wide.shape, "| 与原表一致:", wide.equals(mini.set_index("patient_id")))

melt 后: (6, 3) —— 2样本×3基因 = 6行
patient_id gene    tpm
     LC001 EGFR 117.93
     LC002 EGFR  22.89
     LC001 KRAS   4.84
     LC002 KRAS  15.81
     LC001 TP53  16.09
     LC002 TP53  46.02

pivot 回宽: (2, 3) | 与原表一致: True


> 💬 **讨论点**：`pivot` 遇重复索引报错；`pivot_table` 自动聚合——报错是好事，逼你想清楚重复值怎么办。
> 注意：`head(2)` 的 patient_id 仍唯一，pivot 照常成功；要触发报错得先制造重复键，如 `mini = pd.concat([mini, mini.head(1)])`（同一患者出现两次）再 melt + pivot。

## Slide 27 · pivot / pivot_table：长转宽

In [19]:
# ── pivot_table：groupby + 透视合体（aggfunc 就是上一模块的 agg） ────────
# Slide 27 · pivot 遇重复值报错；pivot_table 自动聚合
pt = long.pivot_table(index="patient_id", columns="gene",
                      values="tpm", aggfunc="mean")
print("pivot_table:", pt.shape, "| 数值与 pivot 一致:",
      np.allclose(pt.values, wide.values))


pivot_table: (2, 3) | 数值与 pivot 一致: True


## Slide 28 · 完整流程闭环：读 → 合 → 组 → 透视 → 存

In [20]:
# ── 五步流程：本章大结局 ────────────────────────────────────────────────
# Slide 28 · R半程镜像：read_csv → left_join → group_by/summarise → pivot_wider → write_csv
merged = pd.merge(expr_df, lung, on="patient_id", how="left")
assert len(merged) == 28, f"merge 后 {len(merged)} 行，期望 28（expr 为左表）"
print("①② 读+合:", len(merged), "行")

summary = (merged.groupby(["stage", "smoking"], as_index=False)
                 .agg(n=("patient_id", "count"),
                      med_expr=("egfr_expr", "median")))          # ③ 组
wide = summary.pivot(index="stage", columns="smoking",
                     values="med_expr")                      # ④ 透视
print("④ 透视:\n", wide)
summary.to_csv("summary_demo.csv", index=False)               # ⑤ 存
print("⑤ 存: summary_demo.csv（index=False，别忘了）")

①② 读+合: 28 行
④ 透视:
 smoking    no    yes
stage               
I        3.02    NaN
II       3.95   5.71
IIIA     4.66   7.88
IIIB      NaN   7.45
IV        NaN  10.15
⑤ 存: summary_demo.csv（index=False，别忘了）


In [21]:
# ── 方法链版：同一件事，链着写 ──────────────────────────────────────
# Slide 28 · 与分步版结果完全一致；作业与实验课两种写法都算对
chained = (pd.merge(expr_df, lung, on="patient_id", how="left")
           .groupby(["stage", "smoking"], as_index=False)
           .agg(n=("patient_id", "count"), med_expr=("egfr_expr", "median")))
print("方法链版与分步版一致:", chained.equals(summary))


方法链版与分步版一致: True


## 收尾

In [22]:
# ── 清理演示产物 ─────────────────────────────────────────────────────────
Path("summary_demo.csv").unlink(missing_ok=True)
print("演示完成 ✅  下讲预告：第11讲 Matplotlib 基础绘图；本讲的 z 矩阵（Slide 10 单元格）在第12–13讲聚类热图正式上岗")

演示完成 ✅  下讲预告：第11讲 Matplotlib 基础绘图；本讲的 z 矩阵（Slide 10 单元格）在第12–13讲聚类热图正式上岗
